# Design B training on a free GPU (Colab)
Run the cells **in order**. Before you start: Runtime > Change runtime type > **T4 GPU**.

**One-time setup (left sidebar, key icon = Secrets).** Add these secrets and switch on *Notebook access* for each:
- `GITHUB_TOKEN`: a GitHub personal access token that can read your private repo `skin-lesion-triage`.
- `KAGGLE_USERNAME` and `KAGGLE_KEY`: from your Kaggle API token (Kaggle > Settings > API).

Never paste a token into a cell or commit it.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())

## 1. Get the code from GitHub

In [ ]:
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
!git clone https://{token}@github.com/maniodev4-ai/skin-lesion-triage.git
%cd skin-lesion-triage
!ls

## 2. Get the dataset
**Option A (Kaggle download).** If this cell fails with a credentials error, use Option B or C.

In [ ]:
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
!pip -q install kaggle
!mkdir -p data
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data --unzip

**Option B (zip on Google Drive).** Upload the HAM10000 zip to your Drive first, then run this instead of Option A (edit the path).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data
# !unzip -q /content/drive/MyDrive/ham10000.zip -d data

**Option C (run on Kaggle instead of Colab).** In a Kaggle notebook with GPU on, add the dataset *skin-cancer-mnist-ham10000*, then link it here instead of downloading:
`!mkdir -p data && ln -s /kaggle/input/skin-cancer-mnist-ham10000/* data/`

In [ ]:
# check the layout: expect the metadata csv and about 5,000 + 5,015 jpgs
!ls data
!ls data/HAM10000_images_part_1 | wc -l
!ls data/HAM10000_images_part_2 | wc -l

## 3. Rebuild the metadata features with this machine's library versions
The split file and its hash come from your laptop (committed to GitHub). Script 02 first checks that the split was not changed, then refits the preprocessor on the training part only.

In [ ]:
!python src/02_preprocess_features.py

## 4. Train Design B (image + metadata), seed 42
The first run also reads and resizes all images (about 3 to 5 minutes), then each epoch takes well under a minute on a T4.

In [ ]:
!python src/05_design_b_finetune.py --seed 42 --epochs 12

## 5. Ablation: image-only version of Design B (same seed)

In [ ]:
!python src/05_design_b_finetune.py --seed 42 --epochs 12 --use_meta 0

## 6. Two more seeds of the full model (for confidence intervals across runs)

In [ ]:
!python src/05_design_b_finetune.py --seed 43 --epochs 12
!python src/05_design_b_finetune.py --seed 44 --epochs 12

## 7. Download the results, then copy them into your repo
Copy the contents of the zip into the same folders in your project (`outputs/metrics`, `outputs/plots`, `models`), then commit.

In [ ]:
!zip -r design_b_results.zip outputs/metrics outputs/plots models/design_b_*.pt
from google.colab import files
files.download("design_b_results.zip")